# State Farm: VGG16 기준 모델 — GPU 1에폭

ImageNet 사전학습 VGG16의 convolution 특징 추출기는 고정하고,
global average pooling → dropout(0.3) → 10개 클래스 분류기를 학습합니다.
원본 VGG16의 큰 fully connected 분류기는 교체합니다.

* 데이터: 공식 `state-farm-distracted-driver-detection` 대회 데이터.
* 검증: 운전자(subject) 기준 80/20 그룹 분리, seed=42. 이미지 비율은 정확히 80/20이 아닐 수 있습니다.
* 입력: 224×224, ImageNet 정규화. 좌우 행동 클래스가 바뀔 수 있어 좌우 반전을 사용하지 않습니다.
* 학습: GPU 필수, 1에폭, batch 32, AdamW, learning rate 0.001.
* 지표: 검증 정확도(높을수록 좋음), multiclass Log Loss(낮을수록 좋음).

Kaggle에서 대회 규칙에 동의하고 공식 대회 데이터를 연결해야 합니다.
GPU와 Internet을 켜세요. 인터넷이 꺼져 있으면 공식 torchvision 가중치
`vgg16-397923af.pth`를 입력 데이터에 첨부할 수 있습니다.
Kaggle에 라벨이 없는 test 데이터는 이 실험의 검증에 사용하지 않습니다.
이 파일에는 실행 결과가 미리 채워져 있지 않습니다.


In [ ]:
import gc
import hashlib
import json
import os
import random
import re
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

SEED = 42
EPOCHS = 1
BATCH_SIZE = 32
NUM_WORKERS = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
CLASS_NAMES = [f'c{i}' for i in range(10)]
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/vgg16_baseline')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 필요합니다. Kaggle Settings → Accelerator에서 GPU를 선택하세요.')
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda:0')
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
       'seed': SEED, 'epochs': EPOCHS})


## 공식 메타데이터와 이미지 찾기
압축 해제된 이미지와 ZIP 형태 모두 지원합니다.
ZIP 이미지는 필요한 항목만 읽으며, 라벨 없는 test 이미지를 압축 해제하지 않습니다.


In [ ]:
def read_driver_metadata(root):
    csv_paths = sorted(root.rglob('driver_imgs_list.csv'))
    if len(csv_paths) > 1:
        raise RuntimeError(f'메타데이터가 여러 개입니다. 공식 대회 입력만 연결하세요: {csv_paths}')
    if csv_paths:
        return pd.read_csv(csv_paths[0]), str(csv_paths[0])
    candidates = []
    for archive in sorted(root.rglob('*.zip')):
        with zipfile.ZipFile(archive) as zf:
            members = [n for n in zf.namelist() if Path(n).name == 'driver_imgs_list.csv']
            for member in members:
                candidates.append((archive, member))
    if len(candidates) != 1:
        raise FileNotFoundError('driver_imgs_list.csv를 하나만 찾을 수 있어야 합니다. 공식 대회 데이터를 연결하세요.')
    archive, member = candidates[0]
    with zipfile.ZipFile(archive) as zf, zf.open(member) as file:
        return pd.read_csv(file), f'{archive}!{member}'

metadata, metadata_source = read_driver_metadata(INPUT_ROOT)
required = {'subject', 'classname', 'img'}
if not required.issubset(metadata.columns):
    raise ValueError(f'필수 열 누락: {required - set(metadata.columns)}')
metadata = metadata[['subject', 'classname', 'img']].copy()
if metadata.isna().any().any():
    raise ValueError('메타데이터에 결측값이 있습니다.')
if set(metadata['classname']) != set(CLASS_NAMES):
    raise ValueError('c0부터 c9까지 10개 클래스가 모두 필요합니다.')
if metadata.duplicated(['classname', 'img']).any() or metadata['img'].duplicated().any():
    raise ValueError('중복 이미지 식별자가 있습니다.')
metadata['label'] = metadata['classname'].str[1:].astype(int)

# Store only locators, never image bytes. Each worker opens its own ZIP handles.
wanted = set(zip(metadata['classname'], metadata['img']))
image_index = {}

def add_image(key, locator):
    if key not in wanted:
        return
    if key in image_index:
        raise RuntimeError(f'이미지 중복 입력: {key}. 같은 데이터를 두 번 연결하지 마세요.')
    image_index[key] = locator

for image_path in sorted(INPUT_ROOT.rglob('*.jpg')):
    key = (image_path.parent.name, image_path.name)
    add_image(key, ('file', str(image_path), ''))

# Prefer a complete extracted copy. Otherwise supplement from ZIP archives.
if len(image_index) != len(wanted):
    found_on_disk = set(image_index)
    archive_pattern = re.compile(r'(?:^|/)(c[0-9])/([^/]+\.jpg)$', re.IGNORECASE)
    for archive in sorted(INPUT_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(archive) as zf:
            for member in zf.namelist():
                match = archive_pattern.search(member.replace('\\', '/'))
                if match:
                    key = (match.group(1).lower(), match.group(2))
                    if key not in found_on_disk:
                        add_image(key, ('zip', str(archive), member))
missing = wanted - set(image_index)
if missing:
    raise FileNotFoundError(f'{len(missing)}개 학습 이미지 누락. 예시: {sorted(missing)[:5]}')
metadata_sha256 = hashlib.sha256(metadata.to_csv(index=False).encode()).hexdigest()
print({'metadata': metadata_source, 'rows': len(metadata),
       'drivers': metadata['subject'].nunique(), 'images_found': len(image_index)})


In [ ]:
def split_by_driver(frame):
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_ids, val_ids = next(splitter.split(frame, frame['label'], groups=frame['subject']))
    train = frame.iloc[train_ids].reset_index(drop=True)
    val = frame.iloc[val_ids].reset_index(drop=True)
    if set(train['subject']) & set(val['subject']):
        raise RuntimeError('운전자 누출이 발견되었습니다.')
    if set(train['img']) & set(val['img']):
        raise RuntimeError('이미지 누출이 발견되었습니다.')
    expected = set(range(10))
    if set(train['label']) != expected or set(val['label']) != expected:
        raise RuntimeError('학습과 검증에 각각 10개 클래스가 모두 있어야 합니다.')
    return train, val

train_df, val_df = split_by_driver(metadata)
train_df.to_csv(OUTPUT_ROOT / 'train_split.csv', index=False)
val_df.to_csv(OUTPUT_ROOT / 'validation_split.csv', index=False)
split_report = {
    'method': 'GroupShuffleSplit(subject)', 'seed': SEED,
    'train_images': len(train_df), 'validation_images': len(val_df),
    'train_drivers': sorted(train_df['subject'].unique().tolist()),
    'validation_drivers': sorted(val_df['subject'].unique().tolist()),
    'class_order': CLASS_NAMES, 'metadata_sha256': metadata_sha256,
}
(OUTPUT_ROOT / 'split_report.json').write_text(json.dumps(split_report, indent=2))
print(json.dumps(split_report, indent=2))
display(pd.DataFrame({
    'train': train_df['classname'].value_counts(),
    'validation': val_df['classname'].value_counts(),
}).reindex(CLASS_NAMES))


In [ ]:
class DriverImages(Dataset):
    def __init__(self, frame, image_index, transform):
        self.frame = frame.reset_index(drop=True)
        self.image_index = image_index
        self.transform = transform
        self._archives = {}
        self._pid = None

    def __len__(self):
        return len(self.frame)

    def __getstate__(self):
        state = self.__dict__.copy()
        state['_archives'] = {}
        state['_pid'] = None
        return state

    def close(self):
        for archive in self._archives.values():
            archive.close()
        self._archives = {}

    def __getitem__(self, index):
        if self._pid != os.getpid():
            self.close()
            self._pid = os.getpid()
        row = self.frame.iloc[index]
        kind, path, member = self.image_index[(row['classname'], row['img'])]
        if kind == 'file':
            with Image.open(path) as image:
                image = image.convert('RGB')
        else:
            if path not in self._archives:
                self._archives[path] = zipfile.ZipFile(path)
            with self._archives[path].open(member) as file:
                with Image.open(file) as image:
                    image = image.convert('RGB')
        return self.transform(image), int(row['label'])

normalization = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(), normalization,
])
validation_transform = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(), normalization,
])
train_dataset = DriverImages(train_df, image_index, train_transform)
val_dataset = DriverImages(val_df, image_index, validation_transform)
# Inspect image decoding before starting workers or downloading weights.
for dataset in (train_dataset, val_dataset):
    x, y = dataset[0]
    assert tuple(x.shape) == (3, 224, 224) and 0 <= y < 10
    dataset.close()

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
                   worker_init_fn=seed_worker, persistent_workers=NUM_WORKERS > 0)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)


In [ ]:
weights_path = Path(torch.hub.get_dir()) / 'checkpoints/vgg16-397923af.pth'
attached_weights = sorted(INPUT_ROOT.rglob('vgg16-397923af.pth'))
if weights_path.is_file() or attached_weights:
    source = weights_path if weights_path.is_file() else attached_weights[0]
    digest = hashlib.sha256()
    with source.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    if not digest.hexdigest().startswith('397923af'):
        raise RuntimeError('VGG16 가중치의 공식 SHA256 접두사가 일치하지 않습니다.')
    original_vgg = models.vgg16(weights=None)
    original_vgg.load_state_dict(torch.load(source, map_location='cpu', weights_only=True))
else:
    try:
        original_vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    except Exception as exc:
        raise RuntimeError('사전학습 가중치 로드 실패. Internet을 켜거나 공식 가중치를 입력에 첨부하세요.') from exc

class VGG16Baseline(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.features = features
        self.features.requires_grad_(False)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 10))

    def forward(self, x):
        with torch.no_grad():
            x = self.pool(self.features(x)).flatten(1)
        return self.classifier(x)

model = VGG16Baseline(original_vgg.features).to(device)
del original_vgg
gc.collect()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE,
                             weight_decay=WEIGHT_DECAY)
amp_enabled = True
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print({'trainable_parameters': trainable_parameters,
       'total_parameters': sum(p.numel() for p in model.parameters()),
       'weights': 'VGG16_Weights.IMAGENET1K_V1', 'mixed_precision': amp_enabled})


In [ ]:
def evaluate(model, loader):
    model.eval()
    targets, probabilities = [], []
    loss_sum = 0.0
    with torch.inference_mode():
        for images, labels in tqdm(loader, desc='Validation'):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
            loss_sum += criterion(logits.float(), labels).item() * len(labels)
            # Float64 probabilities keep sklearn clipping and CE numerically comparable.
            probabilities.append(logits.double().softmax(dim=1).cpu().numpy())
            targets.append(labels.cpu().numpy())
    targets = np.concatenate(targets)
    probabilities = np.concatenate(probabilities)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('검증 확률에 NaN 또는 Inf가 있습니다.')
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-8)
    accuracy = accuracy_score(targets, probabilities.argmax(axis=1))
    multiclass_log_loss = log_loss(targets, probabilities, labels=list(range(10)))
    cross_entropy = loss_sum / len(targets)
    if not np.isclose(cross_entropy, multiclass_log_loss, rtol=1e-5, atol=1e-5):
        raise RuntimeError(f'CE와 Log Loss 불일치: {cross_entropy}, {multiclass_log_loss}')
    return targets, probabilities, float(accuracy), float(multiclass_log_loss)

started = time.perf_counter()
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    model.features.eval()
    train_loss_sum = 0.0
    train_correct = 0
    train_seen = 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits.float(), labels)
        if not torch.isfinite(loss):
            raise RuntimeError('학습 손실이 유한하지 않습니다.')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss_sum += loss.item() * len(labels)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_seen += len(labels)
    targets, probabilities, val_accuracy, val_log_loss = evaluate(model, val_loader)
    row = {'epoch': epoch, 'train_loss': train_loss_sum / train_seen,
           'train_accuracy': train_correct / train_seen,
           'validation_accuracy': val_accuracy, 'validation_log_loss': val_log_loss}
    history.append(row)
    print(json.dumps(row, indent=2))

elapsed = time.perf_counter() - started
np.testing.assert_array_equal(targets, val_df['label'].to_numpy())
metrics = {
    **history[-1], 'epochs_completed': EPOCHS, 'seed': SEED,
    'gpu': torch.cuda.get_device_name(0), 'elapsed_seconds': elapsed,
    'train_images': len(train_df), 'validation_images': len(val_df),
    'weights': 'VGG16_Weights.IMAGENET1K_V1', 'feature_extractor_frozen': True,
    'head': 'GAP -> Dropout(0.3) -> Linear(512,10)',
    'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'trainable_parameters': trainable_parameters,
    'metadata_sha256': metadata_sha256,
}
predictions = val_df[['subject', 'classname', 'img', 'label']].copy()
for i, name in enumerate(CLASS_NAMES):
    predictions[name] = probabilities[:, i]
predictions.to_csv(OUTPUT_ROOT / 'validation_predictions.csv', index=False)
pd.DataFrame(confusion_matrix(targets, probabilities.argmax(axis=1), labels=list(range(10))),
             index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUTPUT_ROOT / 'confusion_matrix.csv')
pd.DataFrame(history).to_csv(OUTPUT_ROOT / 'history.csv', index=False)
torch.save({'model_state_dict': model.state_dict(), 'metrics': metrics,
            'split_report': split_report}, OUTPUT_ROOT / 'vgg16_baseline.pt')
# Write completion marker last, after all required artifacts have been saved.
(OUTPUT_ROOT / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(f'Validation Accuracy: {val_accuracy:.4%}')
print(f'Validation Log Loss: {val_log_loss:.6f}')
print(f'Outputs: {OUTPUT_ROOT}')


## 결과 해석
위의 마지막 셀과 `vgg16_baseline/metrics.json`이 실제 실행 결과입니다.
운전자를 분리한 holdout 점수이므로 이미지 무작위 분리 점수나 Kaggle leaderboard 점수와
같은 조건의 결과가 아닙니다. 1에폭은 실행 가능성과 초기 성능을 확인하는 기준이며,
이후 검증 결과를 보고 VGG block5 미세조정, 학습률, 증강 등을 비교할 수 있습니다.
